# cloudposterior: Monitoring

Monitor MCMC sampling from your phone or any browser. Three options:

- **`notify=True`** -- push notifications via [ntfy](https://ntfy.sh) (sampling start + completion). Works locally and remotely.
- **`notify="my-topic"`** -- ntfy with a fixed topic name you can resubscribe to across runs.
- **`notify="dashboard"`** -- live progress dashboard in your browser with per-chain stats updating in real time. Requires `remote=True`.

> Run this notebook locally to see the QR codes and links.

In [1]:
import numpy as np
import pandas as pd
import pymc as pm

import cloudposterior as cp

In [2]:
df = pd.read_csv(pm.get_data("radon.csv"))

with pm.Model(name="radon_intercepts", coords={"county": df.county.unique()}) as radon:
    mu_a = pm.Normal("mu_a", mu=0, sigma=5)
    sigma_a = pm.HalfNormal("sigma_a", sigma=2)
    a_raw = pm.Normal("a_raw", mu=0, sigma=1, dims="county")
    a = pm.Deterministic("a", mu_a + sigma_a * a_raw, dims="county")
    b_floor = pm.Normal("b_floor", mu=0, sigma=5)
    mu = a[df.county_code.values] + b_floor * df.floor.values
    sigma_y = pm.HalfNormal("sigma_y", sigma=2)
    pm.Normal("obs", mu=mu, sigma=sigma_y, observed=df.log_radon.values)

## Push notifications (local or remote)

Pass `notify=True` to get push notifications when sampling starts and completes. A link and QR code are printed -- scan it with the [ntfy app](https://ntfy.sh) to subscribe. Works for both local and remote runs.

For private notifications, point to your own [ntfy server](https://docs.ntfy.sh/install/) with `notify={"server": "https://ntfy.example.com"}`.

In [3]:
with cp.cloud(radon, cache=False, notify=True):
    idata = pm.sample(draws=2000, tune=1000, chains=4)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [radon_intercepts::mu_a, radon_intercepts::sigma_a, radon_intercepts::a_raw, radon_intercepts::b_floor, radon_intercepts::sigma_y]


Output()

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 2 seconds.


## Custom topic

Use a fixed topic name so you can subscribe once and reuse it across runs. Anyone with the topic name can see your notifications -- pick something unique.

In [4]:
with cp.cloud(radon, cache="disk", notify="my-radon-project"):
    idata = pm.sample(draws=2000, tune=1000, chains=4)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [radon_intercepts::mu_a, radon_intercepts::sigma_a, radon_intercepts::a_raw, radon_intercepts::b_floor, radon_intercepts::sigma_y]


Output()

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 2 seconds.


## Live dashboard (remote)

When running remotely, `notify=True` opens a live progress page hosted on the cloud VM. A URL and QR code are printed -- open it on your phone or any browser. The page shows per-chain progress bars, divergence counts, and speed, updating every second.

In [5]:
with cp.cloud(radon, remote=True, cache=False, notify=True):
    idata = pm.sample(draws=100000, tune=1000, chains=4)